# IMS Feature Engineering

This notebook creates the first preprocessing pipeline for the IMS bearing dataset. It converts each timestamped vibration file into per-channel statistical and frequency-domain features.

Default output:

```text
data/processed/ims_features.csv
```

For smoke tests, set `IMS_FEATURE_MAX_FILES` to a small integer and `IMS_FEATURE_WRITE_OUTPUT=0` before execution. Without those variables, the notebook processes all discovered files and writes the CSV.

In [1]:
from __future__ import annotations

from pathlib import Path
import math
import os
import re
from typing import Iterable

import numpy as np
import pandas as pd
from scipy.fft import rfft, rfftfreq
from scipy.stats import kurtosis, skew

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
IMS_DIR = RAW_DIR / "IMS" / "IMS"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FEATURE_OUTPUT_PATH = PROCESSED_DIR / "ims_features.csv"

SAMPLING_RATE_HZ = 20_000
EXPECTED_MEASUREMENTS_PER_FILE = 20_480
EXPECTED_EXPERIMENTS = {
    "1st_test": {"expected_files": 2156, "expected_channels": 8},
    "2nd_test": {"expected_files": 984, "expected_channels": 4},
    "3rd_test": {"expected_files": 4448, "expected_channels": 4},
}

FEATURE_FILE_LIMIT = os.getenv("IMS_FEATURE_MAX_FILES")
FEATURE_FILE_LIMIT = int(FEATURE_FILE_LIMIT) if FEATURE_FILE_LIMIT else None
WRITE_OUTPUT = os.getenv("IMS_FEATURE_WRITE_OUTPUT", "1").strip().lower() not in {"0", "false", "no"}

PROJECT_ROOT, IMS_DIR, FEATURE_OUTPUT_PATH, FEATURE_FILE_LIMIT, WRITE_OUTPUT

(WindowsPath('C:/Users/Balsem/Desktop/GMAO/ai-service'),
 WindowsPath('C:/Users/Balsem/Desktop/GMAO/ai-service/data/raw/IMS/IMS'),
 WindowsPath('C:/Users/Balsem/Desktop/GMAO/ai-service/data/processed/ims_features.csv'),
 None,
 True)

## 1. Discover Measurement Files

In [2]:
TIMESTAMP_FORMATS = [
    "%Y.%m.%d.%H.%M.%S",
    "%Y-%m-%d-%H-%M-%S",
    "%Y_%m_%d_%H_%M_%S",
]


def safe_rel(path: Path) -> str:
    try:
        return path.resolve().relative_to(PROJECT_ROOT).as_posix()
    except ValueError:
        return path.resolve().as_posix()


def parse_timestamp_from_name(path: Path):
    candidates = [path.name, path.stem]
    for candidate in dict.fromkeys(candidates):
        for fmt in TIMESTAMP_FORMATS:
            try:
                return pd.Timestamp(pd.to_datetime(candidate, format=fmt))
            except (TypeError, ValueError):
                pass
    loose_match = re.search(r"(\d{4})[._-](\d{2})[._-](\d{2})[._-](\d{2})[._-](\d{2})[._-](\d{2})", path.name)
    if loose_match:
        return pd.Timestamp("-".join(loose_match.groups()[:3]) + " " + ":".join(loose_match.groups()[3:]))
    return pd.NaT


def find_experiment_dir(experiment: str) -> Path | None:
    candidates = [
        IMS_DIR / experiment,
        RAW_DIR / "IMS" / experiment,
        RAW_DIR / experiment,
    ]
    for candidate in candidates:
        if candidate.is_dir():
            return candidate
    return None


def measurement_files(experiment_dir: Path | None) -> list[Path]:
    if experiment_dir is None or not experiment_dir.exists():
        return []
    ignored_suffixes = {".zip", ".rar", ".7z", ".pdf", ".md", ".csv"}
    files = []
    for path in experiment_dir.rglob("*"):
        if not path.is_file():
            continue
        if "__MACOSX" in path.parts or path.name.startswith((".", "._")):
            continue
        if path.suffix.lower() in ignored_suffixes:
            continue
        files.append(path)
    return sorted(files, key=lambda item: (parse_timestamp_from_name(item), item.name))


discovery_rows = []
files_by_experiment: dict[str, list[Path]] = {}
for experiment, expected in EXPECTED_EXPERIMENTS.items():
    experiment_dir = find_experiment_dir(experiment)
    files = measurement_files(experiment_dir)
    files_by_experiment[experiment] = files
    discovery_rows.append(
        {
            "experiment": experiment,
            "experiment_dir": safe_rel(experiment_dir) if experiment_dir else None,
            "files_found": len(files),
            "expected_files": expected["expected_files"],
            "extra_files_vs_expected": max(len(files) - expected["expected_files"], 0),
            "missing_files_vs_expected": max(expected["expected_files"] - len(files), 0),
            "expected_channels": expected["expected_channels"],
        }
    )

discovery = pd.DataFrame(discovery_rows)
discovery

,experiment,experiment_dir,files_found,expected_files,extra_files_vs_expected,missing_files_vs_expected,expected_channels
0,1st_test,data/raw/IMS/IMS/1st_test,2156,2156,0,0,8
1,2nd_test,data/raw/IMS/IMS/2nd_test,984,984,0,0,4
2,3rd_test,data/raw/IMS/IMS/3rd_test,6324,4448,1876,0,4


## 2. Channel-to-Bearing Mapping

In [3]:
def channel_metadata(experiment: str, channel_index: int) -> dict[str, int | str | None]:
    channel_number = channel_index + 1
    if experiment == "1st_test":
        bearing = (channel_index // 2) + 1
        axis = "x" if channel_index % 2 == 0 else "y"
        return {"sensor_channel": channel_number, "bearing": bearing, "axis": axis}
    bearing = channel_number
    return {"sensor_channel": channel_number, "bearing": bearing, "axis": None}


channel_mapping_rows = []
for experiment, expected in EXPECTED_EXPERIMENTS.items():
    for channel_index in range(expected["expected_channels"]):
        channel_mapping_rows.append({"experiment": experiment, **channel_metadata(experiment, channel_index)})

channel_mapping = pd.DataFrame(channel_mapping_rows)
channel_mapping

,experiment,sensor_channel,bearing,axis
0,1st_test,1,1,x
1,1st_test,2,1,y
2,1st_test,3,2,x
3,1st_test,4,2,y
4,1st_test,5,3,x
5,1st_test,6,3,y
6,1st_test,7,4,x
7,1st_test,8,4,y
8,2nd_test,1,1,NaN
9,2nd_test,2,2,NaN


## 3. Feature Extraction Functions

In [4]:
def read_measurement_file(path: Path) -> np.ndarray:
    values = np.loadtxt(path)
    if values.ndim == 1:
        values = values.reshape(-1, 1)
    return values


def spectral_features(values: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    centered = values - np.mean(values, axis=0, keepdims=True)
    spectrum = np.abs(rfft(centered, axis=0))
    frequencies = rfftfreq(values.shape[0], d=1 / SAMPLING_RATE_HZ)
    energy = np.sum(spectrum**2, axis=0) / values.shape[0]
    if spectrum.shape[0] <= 1:
        return energy, np.zeros(values.shape[1])
    dominant_indices = np.argmax(spectrum[1:, :], axis=0) + 1
    return energy, frequencies[dominant_indices]


def feature_matrix(values: np.ndarray) -> dict[str, np.ndarray]:
    values = values.astype(float, copy=False)
    rms = np.sqrt(np.mean(values**2, axis=0))
    peak_abs = np.max(np.abs(values), axis=0)
    spectral_energy, dominant_frequency_hz = spectral_features(values)
    return {
        "rms": rms,
        "standard_deviation": np.std(values, axis=0, ddof=1) if values.shape[0] > 1 else np.zeros(values.shape[1]),
        "peak_to_peak": np.ptp(values, axis=0),
        "kurtosis": kurtosis(values, axis=0, fisher=False, bias=False) if values.shape[0] > 3 else np.full(values.shape[1], np.nan),
        "skewness": skew(values, axis=0, bias=False) if values.shape[0] > 2 else np.full(values.shape[1], np.nan),
        "crest_factor": np.divide(peak_abs, rms, out=np.full_like(rms, np.nan), where=rms != 0),
        "spectral_energy": spectral_energy,
        "dominant_frequency_hz": dominant_frequency_hz,
    }


def extract_features_for_file(experiment: str, path: Path) -> list[dict]:
    values = read_measurement_file(path)
    if values.size == 0:
        raise ValueError("Empty measurement file")
    if not np.isfinite(values).all():
        raise ValueError("Non-finite value found")

    rows = []
    timestamp = parse_timestamp_from_name(path)
    metrics = feature_matrix(values)
    for channel_index in range(values.shape[1]):
        rows.append(
            {
                "timestamp": timestamp,
                "experiment": experiment,
                **channel_metadata(experiment, channel_index),
                "measurements": int(values.shape[0]),
                "source_file": safe_rel(path),
                **{name: float(series[channel_index]) for name, series in metrics.items()},
            }
        )
    return rows

## 4. Build Feature Table

In [5]:
feature_rows = []
error_rows = []

for experiment, files in files_by_experiment.items():
    files_to_process = files[:FEATURE_FILE_LIMIT] if FEATURE_FILE_LIMIT is not None else files
    print(f"{experiment}: processing {len(files_to_process)} of {len(files)} files", flush=True)
    for index, path in enumerate(files_to_process, start=1):
        try:
            feature_rows.extend(extract_features_for_file(experiment, path))
        except Exception as exc:
            error_rows.append(
                {
                    "experiment": experiment,
                    "path": safe_rel(path),
                    "error": f"{type(exc).__name__}: {exc}",
                }
            )
        if index % 250 == 0:
            print(f"  {experiment}: processed {index} files", flush=True)

features = pd.DataFrame(feature_rows)
feature_errors = pd.DataFrame(error_rows)

if not features.empty:
    features = features.sort_values(["experiment", "timestamp", "sensor_channel", "source_file"]).reset_index(drop=True)

features.head()

1st_test: processing 2156 of 2156 files


  1st_test: processed 250 files


  1st_test: processed 500 files


  1st_test: processed 750 files


  1st_test: processed 1000 files


...

  3rd_test: processed 5500 files


  3rd_test: processed 5750 files


  3rd_test: processed 6000 files


  3rd_test: processed 6250 files


,timestamp,experiment,sensor_channel,bearing,axis,measurements,source_file,rms,standard_deviation,peak_to_peak,kurtosis,skewness,crest_factor,spectral_energy,dominant_frequency_hz
0,2003-10-22 12:06:24,1st_test,1,1,x,20480,data/raw/IMS/IMS/1st_test/2003.10.22.12.06.24,0.124614,0.081124,1.108,4.069717,-0.029995,5.777850,67.387750,986.328125
1,2003-10-22 12:06:24,1st_test,2,1,y,20480,data/raw/IMS/IMS/1st_test/2003.10.22.12.06.24,0.117493,0.070650,1.265,6.066925,0.220132,5.966305,51.110429,986.328125
2,2003-10-22 12:06:24,1st_test,3,2,x,20480,data/raw/IMS/IMS/1st_test/2003.10.22.12.06.24,0.130455,0.090650,1.033,3.209830,-0.092080,5.166521,84.145905,986.328125
3,2003-10-22 12:06:24,1st_test,4,2,y,20480,data/raw/IMS/IMS/1st_test/2003.10.22.12.06.24,0.121642,0.077510,0.786,3.292586,-0.053187,4.357044,61.516686,986.328125
4,2003-10-22 12:06:24,1st_test,5,3,x,20480,data/raw/IMS/IMS/1st_test/2003.10.22.12.06.24,0.128887,0.091463,0.896,3.405831,0.034374,3.848331,85.658667,986.328125


## 5. Feature Quality Checks

In [6]:
feature_summary = pd.DataFrame(
    [
        {
            "rows": len(features),
            "experiments": features["experiment"].nunique() if not features.empty else 0,
            "timestamps": features["timestamp"].nunique() if not features.empty else 0,
            "bearings": features["bearing"].nunique() if not features.empty else 0,
            "source_files": features["source_file"].nunique() if not features.empty else 0,
            "feature_file_limit": FEATURE_FILE_LIMIT,
            "write_output": WRITE_OUTPUT,
            "errors": len(feature_errors),
        }
    ]
)
feature_summary

,rows,experiments,timestamps,bearings,source_files,feature_file_limit,write_output,errors
0,46480,3,9464,4,9464,None,True,0


In [7]:
if not features.empty:
    quality_by_experiment = features.groupby("experiment").agg(
        rows=("timestamp", "size"),
        timestamps=("timestamp", "nunique"),
        bearings=("bearing", "nunique"),
        channels=("sensor_channel", "nunique"),
        first_timestamp=("timestamp", "min"),
        last_timestamp=("timestamp", "max"),
    ).reset_index()
else:
    quality_by_experiment = pd.DataFrame()

quality_by_experiment

,experiment,rows,timestamps,bearings,channels,first_timestamp,last_timestamp
0,1st_test,17248,2156,4,8,2003-10-22 12:06:24,2003-11-25 23:39:56
1,2nd_test,3936,984,4,4,2004-02-12 10:32:39,2004-02-19 06:22:39
2,3rd_test,25296,6324,4,4,2004-03-04 09:27:46,2004-04-18 02:42:55


In [8]:
feature_errors

""


## 6. Save Processed Features

In [9]:
if WRITE_OUTPUT:
    PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
    features.to_csv(FEATURE_OUTPUT_PATH, index=False)
    print(f"Saved {len(features):,} feature rows to {FEATURE_OUTPUT_PATH}")
else:
    print("Skipped CSV write because IMS_FEATURE_WRITE_OUTPUT is disabled.")

FEATURE_OUTPUT_PATH

Saved 46,480 feature rows to C:\Users\Balsem\Desktop\GMAO\ai-service\data\processed\ims_features.csv


WindowsPath('C:/Users/Balsem/Desktop/GMAO/ai-service/data/processed/ims_features.csv')